In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import os
import sys
import pickle
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
import spk_feat_cluster_comp_analysis
importlib.reload(spk_feat_cluster_comp_analysis)
from spk_feat_cluster_comp_analysis import (
    plot_transition_deltas_signed_mean,
    plot_lfp_block_comparison,
)
import config

CLUSTER_PKL_DIR = config.SPE1_PICKLE_ROOT + '/cluster_pickles/'
LFP_NPY_DIR     = os.path.join(config.SPE1_DATA_ROOT, 'filt_lfp_recordings')

In [ ]:
df_transitions = pd.read_pickle(CLUSTER_PKL_DIR + 'temporal_transitions.pkl')
peri_results   = pickle.load(open(CLUSTER_PKL_DIR + 'lfp_peri_transition_results.pkl', 'rb'))

# peri_results keys: (cell_id, feat, transition_index)
used_pairs = {(k[0], k[1]) for k, v in peri_results.items() if len(v) == 3}
print(f'{len(df_transitions.groupby(["cell_id","spike_feature"]))} detected transitions')
print(f'{len(peri_results)} transition LFP entries, {len(used_pairs)} unique cell×feat pairs')

In [ ]:
# Panel A: population-level sign-corrected LFP delta, by spike feature
# Sign convention: positive = LFP metric increased when spike cluster increased (low→high)
fig = plot_transition_deltas_signed_mean(peri_results, df_transitions)
plt.show()

In [ ]:
METRICS = ['exponent', 'theta_auc', 'slow_gamma_auc', 'high_gamma_auc', 'total_gamma_auc']
ORD = {'low':0,'mid':1,'high':2,'Low':0,'Mid':1,'High':2}

rows = []
for (cid, feat, t_idx), blocks in peri_results.items():
    bmap = {b['label']: b for b in blocks}
    if 'pre' not in bmap or 'post' not in bmap:
        continue
    tr = df_transitions[(df_transitions.cell_id==cid) &
                        (df_transitions.spike_feature==feat) &
                        (df_transitions.transition_index==t_idx)]
    if tr.empty: continue
    before = ORD.get(str(tr.iloc[0]['cluster_before']).lower(), 1)
    after  = ORD.get(str(tr.iloc[0]['cluster_after']).lower(),  1)
    sign   = 1 if after > before else -1
    row = dict(cell_id=cid, feat=feat, t_idx=t_idx, sign=sign)
    for m in METRICS:
        row[f'd_{m}'] = sign * (bmap['post'][m] - bmap['pre'][m])
    rows.append(row)

df_top = pd.DataFrame(rows)

# Composite score: sum of |normalized delta| across all metrics
for m in METRICS:
    sd = df_top[f'd_{m}'].std()
    df_top[f'n_{m}'] = df_top[f'd_{m}'] / (sd + 1e-12)
df_top['composite'] = df_top[[f'n_{m}' for m in METRICS]].abs().sum(axis=1)
df_top = df_top.sort_values('composite', ascending=False)

print('Top transitions by composite |Δ| across all LFP metrics:')
print(df_top.head(6)[['cell_id','feat','t_idx','sign','composite'] +
                      [f'd_{m}' for m in METRICS]].round(3).to_string())

In [ ]:
# Panel B: exemplar cells — top 4 transitions by composite LFP delta
top_pairs = set(df_top.head(4)[['cell_id','feat','t_idx']].itertuples(index=False, name=None))

peri_top = {k: v for k, v in peri_results.items() if (k[0], k[1], k[2]) in top_pairs}
df_trans_top = df_transitions[
    df_transitions.apply(
        lambda r: (r.cell_id, r.spike_feature, int(r.transition_index)) in top_pairs, axis=1)
]

plot_lfp_block_comparison(peri_top, df_trans_top, CLUSTER_PKL_DIR, rolling_n=50)
plt.show()

**Supplementary Figure X. LFP spectral changes associated with within-neuron spike cluster transitions.**

(A) ...

(B) ...